In [ ]:
import sys, os
# Ноутбук лежит в notebooks/, а utils.py — в src/. Добавляем путь.
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor

from utils import (
    create_bipartite_bayesian_network_cond,
    create_bipartite_bayesian_network_nocond,
    f_2, f_3, f_4, f_5,
    ratio,
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# f_2..f_5 и ratio импортированы из utils выше.
# Здесь — быстрая проверка нормировки и угловых значений.
print("f_2+f_3+f_4+f_5 на сетке:",
      (f_2(np.linspace(0.05, 0.95, 5))
       + f_3(np.linspace(0.05, 0.95, 5))
       + f_4(np.linspace(0.05, 0.95, 5))
       + f_5(np.linspace(0.05, 0.95, 5))).round(6))
print("ratio current (0.5, 0.5) =", ratio(0.5, 0.5, kind="current"))
print("ratio sigmoid (0.5, 0.5) =", ratio(0.5, 0.5, kind="sigmoid"))

In [ ]:
def generate_data(n_students, m_items, random_state=42):
    """
    params:
        n_studetns (int): количество студентов
        m_items (int): количество предметов
    """
    np.random.seed(random_state)
    return np.random.randint(low=2, high=5, size=n_students * m_items).reshape(n_students, m_items)

In [ ]:
def generate_rating_matrix(n_students, m_items, seed=42):
    """
    params:
        n_studetns (int): количество студентов
        m_items (int): количество предметов
    """
    np.random.seed(seed)

    perf = st.beta(a=2, b=2).rvs(size=n_students)
    diff = st.beta(a=2, b=2).rvs(size=m_items)

    rating_matrix = np.zeros((n_students, m_items))

    for i, perf_i in enumerate(perf):
        for j, diff_j in enumerate(diff):
            r = ratio(perf_i, diff_j)   # 1 - diff_j * (1 - perf_i)

            w2 = f_2(r)
            w3 = f_3(r)
            w4 = f_4(r)
            w5 = f_5(r)
            w_sum = w2 + w3 + w4 + w5

            pvals = [w2 / w_sum, w3 / w_sum, w4 / w_sum, w5 / w_sum]
            marks = np.random.multinomial(n=1, pvals=pvals, size=1).flatten()

            rating_matrix[i, j] = int(np.argmax(marks)) + 2

    return rating_matrix, perf, diff

In [ ]:
def eval_stats(rm_shapes, model, model_params):
    """
    params:
        rm_shapes (list((n1, m1), ..., (nk, mk)): shape'ы матриц рейтинга
        model: модель
        model_params dict(): параметры модели
    """
    mae_stats = []
    max_error_stats = []
    for rm_shape in rm_shapes:
        n_students, m_items = rm_shape
        rm, perf, diff = generate_rating_matrix(n_students, m_items)

        trace, _ = model(ratings_matrix=rm, **model_params)

        mae = np.mean(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))
        max_error = np.max(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))

        mae_stats.append(mae)
        max_error_stats.append(max_error)

    return mae_stats, max_error_stats

In [ ]:
rm_shapes = [(5, 5), (10, 10), (20, 20)]
params = {
    'student_alpha': 2,
    'student_beta': 2,
    'item_alpha': 2,
    'item_beta': 2,
    'draws':1000,
    'tune':2000,
    'chains':6,
    'cores':6,

}

cond_stats = eval_stats(rm_shapes, create_bipartite_bayesian_network_cond, model_params=params)

In [ ]:
rm_shapes = [(5, 5), (10, 10), (20, 20)]
params = {
    'student_alpha': 2,
    'student_beta': 2,
    'item_alpha': 2,
    'item_beta': 2,
    'draws':1000,
    'tune':2000,
    'chains':6,
    'cores':6
}

no_cond_stats = eval_stats(rm_shapes, create_bipartite_bayesian_network_nocond, model_params=params)

In [ ]:
print("Статистики для модели с условием на отличников и двоечников:")
print(cond_stats)
print("Статистики для модели без условия:")
print(no_cond_stats)

In [ ]:
rm, perf, diff = generate_rating_matrix(10, 5)

trace, model = create_bipartite_bayesian_network_cond(ratings_matrix=rm, **params)

In [ ]:
# смотрим как модель отображает красиво зависимости

pm.model_to_graphviz(model)

In [ ]:
# смотрим на граф вычислений (зависимости, которые учитываются в pyMC)

pytensor.dprint(model.logp())

In [ ]:
"""
MAE реализации сл величины от среднего оценки
"""
np.mean(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))

In [ ]:
"""
Максимальное отклонение реализации сл величины от среднего оценки
"""
np.max(np.abs(az.summary(trace)['mean'] - np.concatenate([perf, diff], axis=0)))

In [ ]:
def show_trace(trace, figsize):
    ncols, nrows = figsize
    fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(14, 30))
    
    az.plot_trace(trace,
                compact=False,
                legend=True,
                axes=axes)
    axes = axes.flatten()
    for i in range(0, len(axes), 2):
        axes[i].set_xlim(0, 1)
    return fig, axes

In [ ]:
fig, axes = show_trace(trace, (2, 15))
rvals = np.concatenate([perf, diff], axis=0)
for i in range(0, len(rvals)):
    axes[i * 2].axvline(rvals[i])
plt.tight_layout()
plt.show()